# Migration of the WavLM Base features and of the original tables

Run after `notebooks/01_environment_check.ipynb`, at the same commit and on
the same kind of runtime: a Colab runtime with a T4 GPU. A CPU runtime is
not enough. Most of the time is Drive I/O, and every step can be run again
(a copy that already exists with the right content is kept).

It reads the original folders and writes only under the new root
(`config.ROOT`). Nothing under the original folders is written, moved or
deleted.

1. free space on the new root (hard stop below the required amount);
2. the ASVspoof 2019 LA archive and the two official ASV score files, with their sha256;
3. the unpadded feature shards of the four conditions, copied file by file and verified by sha256, then the per-file contents and the number of files per split;
4. layer 4 of the padded shards of the two conditions of the padded comparison, written into new shards and compared tensor by tensor;
5. the file lists against the copied shards (the order of the list defines the shards);
6. In-the-Wild from the pinned revision (only when the revision is pinned);
7. the original tables and files that later checks read, copied under `migration/original/` with their sha256, and compared by sha256 with the paper archive (`paper_tables.zip` in `<root>/migration/`).

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Google Drive is not mounted"

### Repository at a fixed commit

Set `COMMIT` to the commit of this repository that is being checked: the
full hash or at least its first 7 characters. The repository is public and
is cloned without a token. If the Colab secret `GITHUB_TOKEN` exists and
this notebook has access to it, the clone uses it (a repository that is not
public needs one); the token is not kept in the clone and is replaced by
`***` in everything git prints.

In [ ]:
import subprocess

REPOSITORY = "Nooralabd/silence-shortcut-wavlm-layers"
COMMIT = ""          # the commit to run: the full hash or at least its first 7 characters
WORKDIR = "/content/silence-shortcut-wavlm-layers"

assert len(COMMIT) >= 7 and set(COMMIT) <= set("0123456789abcdef"), \
    "set COMMIT to the commit that is being checked (full hash or at least its first 7 characters)"
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:    # the secret does not exist, or this notebook has no access to it
    token = ""       # a public repository is cloned without a token


def git(*arguments):
    """Run git and print what it prints; the token is never printed."""
    result = subprocess.run(["git", *arguments], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***")
    if output:
        print(output)
    assert result.returncode == 0, "git failed: see the lines above"
    return output


def clone(repository, folder):
    """Clone a repository, or fetch into an existing clone; a token, if there is one, is not kept in it."""
    address = f"https://{token}@github.com/{repository}.git" if token else f"https://github.com/{repository}.git"
    if not os.path.isdir(folder):
        git("clone", "-q", address, folder)
        git("-C", folder, "remote", "set-url", "origin", f"https://github.com/{repository}.git")
    else:
        git("-C", folder, "fetch", "-q", address, "+refs/heads/*:refs/remotes/origin/*")


clone(REPOSITORY, WORKDIR)
git("-C", WORKDIR, "checkout", "-q", COMMIT)
git("-C", WORKDIR, "rev-parse", "HEAD")
%cd {WORKDIR}

In [ ]:
# Installation. torch, torchaudio and torchcodec are the builds that extracted
# the original features: the pinned versions for CUDA 13.0. If anything pinned
# is missing or has another version, everything is installed in one pass and
# the cell always stops for a restart: a kernel never continues after pip has
# run in it (the runtime has imported numpy before this cell, and a module
# that is already loaded stays loaded when pip replaces it on disk).
# On the run after the restart nothing is installed: the cell checks the
# installed versions and the versions of the modules that the kernel loads.
import importlib
import importlib.metadata
import re
import sys

TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
TORCH_FAMILY = ("torch", "torchaudio", "torchcodec")          # installed from TORCH_INDEX
INSTALLED_ONCE = os.path.join(os.path.dirname(WORKDIR), "installed_once.txt")
OTHERS = os.path.join(os.path.dirname(WORKDIR), "requirements_without_torch.txt")
CONSTRAINT = os.path.join(os.path.dirname(WORKDIR), "torch_constraint.txt")

with open(os.path.join(WORKDIR, "requirements.txt"), encoding="utf-8") as fh:
    requirements = [line.strip() for line in fh if line.strip() and not line.lstrip().startswith("#")]
pins = dict(line.split("==") for line in requirements if "==" in line)
BUILDS = {name: pins[name] + "+" + TORCH_INDEX.rsplit("/", 1)[1] for name in TORCH_FAMILY}


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def not_as_pinned():
    """{package: (installed, wanted)} for everything that is missing or has another version."""
    found = {}
    for line in requirements:
        name = re.split(r"[=<>!~ ]", line, maxsplit=1)[0]
        have, want = installed(name), BUILDS.get(name, pins.get(name))
        same = have is not None and (want is None or (have if name in BUILDS else have.split("+")[0]) == want)
        if not same:
            found[name] = (have, want)
    if installed("ssl-silence") is None:
        found["ssl-silence"] = (None, "this repository")
    return found


different = not_as_pinned()
if different:
    assert not os.path.exists(INSTALLED_ONCE), (
        f"the libraries were installed once already and these are still not as pinned: {different}. "
        "Stop: do not restart again")
    imported = sorted(name for name in (*TORCH_FAMILY, "transformers", "ssl_silence") if name in sys.modules)
    assert not imported, (f"{imported} were imported before the installation: "
                          "Runtime > Restart session, then run the notebook again from the top")
    with open(OTHERS, "w", encoding="utf-8") as fh:
        for line in requirements:
            if re.split(r"[=<>!~ ]", line, maxsplit=1)[0] not in TORCH_FAMILY:
                print(line, file=fh)
    with open(CONSTRAINT, "w", encoding="utf-8") as fh:
        for name in TORCH_FAMILY:
            print(f"{name}=={BUILDS[name]}", file=fh)
    # The build is part of the pin (torch==2.11.0+cu130): for a pin without it pip
    # keeps another build of the same version that the runtime ships (+cpu).
    FAMILY_PINS = " ".join(f"{name}=={BUILDS[name]}" for name in TORCH_FAMILY)
    # torchvision of the runtime is built for its own torch and breaks imports
    # with another one; this package does not use it.
    !pip uninstall -q -y torchvision
    !pip install -q {FAMILY_PINS} --index-url {TORCH_INDEX}
    !pip install -q -r {OTHERS} -c {CONSTRAINT}
    !pip install -q --no-deps -e .
    importlib.invalidate_caches()
    # Before the restart: everything on disk is as pinned, so the next run installs nothing.
    different = not_as_pinned()
    assert not different, f"pip did not install these as pinned: {different}: stop"
    with open(INSTALLED_ONCE, "w", encoding="utf-8") as fh:
        print("the pinned libraries were installed in this runtime", file=fh)
    # pip has run in this kernel: it always stops here.
    raise RuntimeError("The pinned libraries were installed. "
                       "Runtime > Restart session, then run the notebook again from the top")

# Nothing was installed in this run. The modules that this kernel loads must
# be the pinned versions too, not only the files on disk.
LOADED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "scikit-learn": "sklearn",
          "torch": "torch", "torchaudio": "torchaudio", "transformers": "transformers"}
for name, module in LOADED.items():
    loaded = importlib.import_module(module).__version__
    assert loaded.split("+")[0] == pins[name], (
        f"{module} {loaded} is loaded in this kernel and the pin is {pins[name]}: it was imported before pip "
        "replaced it. Runtime > Restart session, then run the notebook again from the top")

import torch
import torchaudio
import transformers

print("loaded:", {name: sys.modules[module].__version__ for name, module in LOADED.items()})
print("torch", torch.__version__, "| CUDA of the build:", torch.version.cuda)
print("torchaudio", torchaudio.__version__, "| torchcodec", installed("torchcodec"),
      "| transformers", transformers.__version__)
if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))   # a running kernel does not see a new editable install

In [ ]:
import subprocess
import sys

import pandas as pd

from ssl_silence import config as cfg
from ssl_silence import provenance, verify

pd.set_option("display.max_rows", 200, "display.width", 200)
HEAD = provenance.git_commit()     # the full hash, with "-dirty" if a tracked file was changed
assert HEAD.startswith(COMMIT) and not HEAD.endswith("-dirty"), (HEAD, COMMIT)
print("commit:", HEAD)
print("root:", cfg.ROOT)

In [ ]:
# The quota of the Google account is read from the Drive API (this asks for
# access to the account); the mounted folder does not report it.
QUOTA = cfg.drive_quota()
print("Google Drive: {usage_gb:.1f} GB used of {limit_gb:.1f} GB, {free_gb:.1f} GB free".format(**QUOTA),
      "(GB = 10^9 bytes; required: %d GB free)" % cfg.MIN_FREE_GB)
cfg.require_free_space(quota=QUOTA)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### Mapping file

`legacy_map.yaml` names the original folders, files and labels. It is kept on
Drive only, at `<root>/migration/legacy_map.yaml`, and must have no empty
value (`verify.load_map` stops otherwise).

In [ ]:
MAP_PATH = os.path.join(cfg.migration_dir(), "legacy_map.yaml")
assert os.path.exists(MAP_PATH), f"upload legacy_map.yaml to {MAP_PATH}"
legacy = verify.load_map(MAP_PATH)
FORBIDDEN = legacy["forbidden"]
CHECKS = cfg.checks_dir()
checks = {}


def record(name, table, stop=True):
    """Write a check table to Drive, show it, and stop if a row fails."""
    path = verify.write_check(table, name, CHECKS, FORBIDDEN)
    checks[name] = table
    display(table)
    if stop and "ok" in table.columns:
        verify.require_ok(table, name)
    return path

In [ ]:
from ssl_silence import datasets, metrics, migration, shards

ENCODER = "wavlm-base"
CONDITIONS = list(cfg.conditions())
names = legacy["features"][ENCODER]

### Paper archive

`paper_tables.zip` holds the tables that the paper was written from, under
their original relative paths, with `MANIFEST.csv` (path, rows, sha256). It
is uploaded to `<root>/migration/`. Every table of the archive must have the
sha256 of its manifest. The original tables copied below are compared with
it, because the original folders hold more than one copy of some tables.

In [ ]:
ARCHIVE = os.path.join(cfg.migration_dir(), "paper_tables.zip")
assert os.path.exists(ARCHIVE), f"upload the paper archive to {ARCHIVE}"
print(len(migration.archive_hashes(ARCHIVE)), "tables in the paper archive, all with the sha256 of its manifest")

### What the original shards record

The commit, the audio library and the library versions recorded in the
original shards, for the unpadded shards of every condition and for the
padded shards (they were extracted at different commits of the original
code).

In [ ]:
rows = []
for extraction, folders in (("unpadded", names["unpadded"]), ("padded", names["padded"]["folders"])):
    for condition, folder in folders.items():
        part = migration.shard_provenance(verify.legacy_path(legacy, folder))
        rows.append(part.assign(extraction=extraction, condition=condition))
recorded = pd.concat(rows, ignore_index=True)
record("original_shard_provenance_all", recorded, stop=False)
display(recorded[recorded["key"] == "git_commit"])

### ASVspoof 2019 LA archive and the official ASV score files

In [ ]:
source = verify.legacy_path(legacy, legacy["files"]["asvspoof_zip"])
manifest = migration.copy_files([(source, cfg.la_zip())], cfg.ROOT, progress=True)
scores = metrics.extract_asv_scores(cfg.la_zip(), cfg.asv_scores_dir())
rows = [{"file": migration.relative(path, cfg.ROOT), "bytes": os.path.getsize(path),
         "sha256": provenance.sha256_file(path)} for path in scores.values()]
record("dataset_files", pd.concat([manifest, pd.DataFrame(rows)], ignore_index=True), stop=False)

### Unpadded features: copy and verify

About 47 GB for the four conditions. The copy of a file is compared with its
source by sha256 before the next file is copied.

In [ ]:
manifests = []
for condition in CONDITIONS:
    source = verify.legacy_path(legacy, names["unpadded"][condition])
    target = cfg.feature_dir(ENCODER, "unpadded", condition)
    part = migration.copy_feature_dir(source, target, cfg.ROOT)
    manifests.append(part.assign(extraction="unpadded", condition=condition))
shard_manifest = pd.concat(manifests, ignore_index=True)
record("shard_manifest_unpadded", shard_manifest, stop=False)

The per-file contents of every shard (lengths, flags, labels, ids) against
the original shard, and the number of files per split. The feature tensors
are identical to the source by the sha256 of the files; set `FULL = True` to
also compare every feature tensor (this reads all shards of both sides again).

In [ ]:
FULL = False
keys = shards.PER_FILE_KEYS["unpadded"] + shards.LIST_KEYS
keys += shards.FEATURE_KEYS["unpadded"] if FULL else ()
for condition in CONDITIONS:
    table = verify.compare_feature_dirs(cfg.feature_dir(ENCODER, "unpadded", condition),
                                        verify.legacy_path(legacy, names["unpadded"][condition]),
                                        keys={key: key for key in keys})
    record(f"features_unpadded_{condition}", table)

### Padded features: one layer

Only the layer of the two configurations of the padded comparison is copied.
It is written into new shards with the key names of this package and compared
with that layer of the original tensors.

In [ ]:
padded = names["padded"]
manifests = []
for condition, folder in padded["folders"].items():
    source = verify.legacy_path(legacy, folder)
    target = cfg.feature_dir(ENCODER, "padded", condition)
    part = migration.copy_padded_layer(source, target, cfg.ROOT, padded["keys"], int(padded["layer"]),
                                       ENCODER, condition)
    manifests.append(part.assign(extraction="padded", condition=condition))
    table = verify.compare_feature_dirs(target, source, keys=padded["keys"], extraction="padded",
                                        old_layer=int(padded["layer"]))
    record(f"features_padded_{condition}", table)
record("shard_manifest_padded", pd.concat(manifests, ignore_index=True), stop=False)

### File lists against the copied shards

The order of a file list defines the shards, so the audio ids, labels and
system ids of the list must equal those stored in the shards, file by file.

In [ ]:
datasets.ensure_asvspoof(cfg.la_zip())
splits = ["train", "dev", "eval"]

In-the-Wild is added when its revision is pinned. With the revision pinned
and no sha256 yet, the archive is downloaded once to the new root and its
sha256 is printed: write it into `config.ITW_ZIP_SHA256`, commit, and run
this notebook again at that commit.

In [ ]:
if cfg.ITW_REVISION is None:
    print("config.ITW_REVISION is not set: In-the-Wild is left out of this run")
elif cfg.ITW_ZIP_SHA256 is None:
    digest = datasets.download_itw(cfg.itw_zip())
    print("sha256 of the In-the-Wild archive:", digest)
    print("write it into config.ITW_ZIP_SHA256, commit, and run this notebook again")
else:
    datasets.ensure_itw(cfg.itw_zip(), sha256=cfg.ITW_ZIP_SHA256)
    splits.append("itw")
print("splits checked:", splits)

In [ ]:
rows = []
for split in splits:
    file_list = datasets.items(split)
    datasets.check_items(file_list, split)
    for condition in CONDITIONS:
        row = migration.check_file_list(file_list, cfg.feature_dir(ENCODER, "unpadded", condition), split)
        rows.append({"condition": condition, **row})
record("file_lists_against_shards", pd.DataFrame(rows))

### Original tables and files

The original tables and files named in the mapping file are copied under
`migration/original/` (tables under the name of the table of this package
they are compared with). Their sha256 identify what the later checks read.

Every copied table is then compared by sha256 with the table of the paper
archive at the original path named in the mapping file. A table that differs
from the archive stops the notebook. A table without a counterpart at that
path is listed as "not in archive"; `same_name_in_archive` counts the tables
of the archive with its file name at another path, and
`same_content_in_archive` says whether the archive holds its content at
another path. The per-file measurement tables of the folder entries are not
compared (the archive never holds them).

In [ ]:
target = os.path.join(cfg.migration_dir(), "original")
manifest = migration.copy_original_files(legacy, target, cfg.ROOT, progress=True)
record("original_files_manifest", manifest, stop=False)
display(manifest[manifest["file"].str.contains("cross_consistency")])
record("original_tables_against_archive", migration.compare_with_archive(legacy, target, cfg.ROOT, ARCHIVE))

### Summary

In [ ]:
summary = verify.summarise_checks({name: table for name, table in checks.items() if "ok" in table.columns})
display(summary)
print("check tables written to", CHECKS)
assert summary["ok"].all()
print("In-the-Wild included:", "itw" in splits)